# akk2eng M01 — Kaggle inference (T5-small)

**Frozen reference:** paths below match a **validated M01-C** run on Kaggle. For a fresh run, copy into `kaggle/akk2eng_m01_submission.ipynb` and adjust `MODEL_PATH` / `INPUT_DIR` for your mounts.

**Self-contained** (no `src/` imports). Produces `submission.csv` under `/kaggle/working/`.

1. Attach the **competition** dataset.
2. (Recommended) Attach a **second dataset** containing your **local fine-tuned** folder (`config.json`, tokenizer, weights) from `outputs/m01_t5/`. Set `MODEL_INPUT` to that path.
3. If no fine-tuned folder is present, the notebook falls back to **base** `google-t5/t5-small` (for smoke tests only).

If `INPUT_DIR` is wrong, list `/kaggle/input` and set `INPUT_DIR` to the folder that contains `test.csv`.

In [1]:
import os
import random

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
# Kaggle dataset that contains your fine-tuned export (directory with config.json)
MODEL_PATH = "/kaggle/input/models/michael1232/akk2eng-m01-model-zip/pytorch/default/1"
BASE_MODEL = "google-t5/t5-small"
T5_TASK_PREFIX = "translate Old Assyrian Akkadian to English: "
MAX_INPUT_LENGTH = 512
MAX_NEW_TOKENS = 256

TEST_CSV = os.path.join(INPUT_DIR, "test.csv")
SUBMISSION_PATH = "/kaggle/working/submission.csv"

if not os.path.isfile(TEST_CSV):
    raise FileNotFoundError(
        f"Missing {TEST_CSV}. List os.listdir('/kaggle/input') and set INPUT_DIR."
    )


def resolve_model_path():
    cfg = os.path.join(MODEL_PATH, "config.json")
    if os.path.isfile(cfg):
        return MODEL_PATH, True
    return BASE_MODEL, False


model_path, from_finetuned = resolve_model_path()
print("TEST_CSV:", TEST_CSV)
print("MODEL_PATH:", model_path, "| fine-tuned:", from_finetuned)
print("OUT:", SUBMISSION_PATH)

TEST_CSV: /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
MODEL_PATH: /kaggle/input/models/michael1232/akk2eng-m01-model-zip/pytorch/default/1 | fine-tuned: True
OUT: /kaggle/working/submission.csv


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("device:", device)

test_df = pd.read_csv(TEST_CSV)
if "id" not in test_df.columns or "transliteration" not in test_df.columns:
    raise ValueError("test.csv must contain id and transliteration")

texts = test_df["transliteration"].fillna("").astype(str).tolist()
prefixed = [f"{T5_TASK_PREFIX}{t}" for t in texts]

translations = []
tok_lens = []
batch_size = 8

with torch.inference_mode():
    for i in range(0, len(prefixed), batch_size):
        chunk = prefixed[i : i + batch_size]
        enc = tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
        )
        for row in gen:
            tok_lens.append(int((row != tokenizer.pad_token_id).sum().item()))
        translations.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))

avg_len = sum(tok_lens) / len(tok_lens) if tok_lens else 0.0
print("rows:", len(test_df))
print("average output token length:", round(avg_len, 2))

print("Sample predictions:")
for j in range(min(8, len(texts))):
    s = texts[j][:200] + ("…" if len(texts[j]) > 200 else "")
    p = translations[j][:200] + ("…" if len(translations[j]) > 200 else "")
    print(f"  [{j}] {s!r} -> {p!r}")

submission = pd.DataFrame({"id": test_df["id"], "translation": translations})
submission.to_csv(SUBMISSION_PATH, index=False)

check = pd.read_csv(SUBMISSION_PATH)
assert list(check.columns) == ["id", "translation"]
assert len(check) == len(test_df)
print(check.head())

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

device: cpu
rows: 4
average output token length: 59.5
Sample predictions:
  [0] 'um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-kam' -> 'From Kn-iya to Aaqil: I have been able to make purchases and make purchases.'
  [1] 'i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au kà-ru-um kà-ni-ia i-lá-qé' -> "To Aur, from Uum: I have seized a tablet with my seals. Witnessed by Aur, by the merchant, by the merchant, by the merchant. Witnessed by Aur's father, by the merchant, by the merchant."
  [2] 'ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-ma a-dí-ni lá i-dí-in ma-lá KÙ.AN na-áa-ú ni-bi„-it a-aí-im au-um-au ú au-mì a-bi…' -> 'To Tamasaya from Am: If you have not paid for the goods, then you must pay the silver. If you have not paid the silver, then you must pay the 

In [5]:
import os
print(os.path.isfile("/kaggle/working/submission.csv"))


True
